# 01_clean_kaggle
Cleans raw Kaggle F1 CSVs from DBFS Volume and writes Parquet to processed Volume.

**Run order:** This notebook must run first — all other notebooks depend on its output.

**Before running:** Make sure all 14 CSVs are uploaded to `/Volumes/workspace/f1_pipeline/raw_kaggle/`

## Cell 1 — One-time setup: Create schema + volumes
Only needs to run once. Safe to re-run (uses IF NOT EXISTS).

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.f1_pipeline")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.raw_kaggle")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.raw_jolpica")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.raw_openf1")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.processed_kaggle")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.processed_jolpica")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.processed_openf1")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.f1_pipeline.processed_staging")

print("✓ Schema and volumes ready")

✓ Schema and volumes ready


## Cell 2 — Verify CSVs are uploaded
Run this after uploading your 14 CSVs. You should see all 14 files listed.

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/f1_pipeline/raw_kaggle/")
for f in files:
    print(f"  {f.name}  ({f.size:,} bytes)")
print(f"\nTotal: {len(files)} files (expecting 14)")

  circuits.csv  (10,104 bytes)
  constructor_results.csv  (219,365 bytes)
  constructor_standings.csv  (317,206 bytes)
  constructors.csv  (17,478 bytes)
  driver_standings.csv  (883,771 bytes)
  drivers.csv  (94,367 bytes)
  lap_times.csv  (17,622,395 bytes)
  pit_stops.csv  (443,719 bytes)
  qualifying.csv  (465,231 bytes)
  races.csv  (164,344 bytes)
  results.csv  (1,721,961 bytes)
  seasons.csv  (4,594 bytes)
  sprint_results.csv  (24,732 bytes)
  status.csv  (2,136 bytes)

Total: 14 files (expecting 14)


## Cell 3 — Main script (clean_kaggle)
Paste and run. Databricks provides `spark` automatically — SparkSession block removed.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType,
)

# ── Configuration ────────────────────────────────────────────────
MODE = "databricks"  # "databricks" | "glue"

PATHS = {
    "databricks": {
        "raw":       "/Volumes/workspace/f1_pipeline/raw_kaggle",
        "processed": "/Volumes/workspace/f1_pipeline/processed_kaggle",
    },
    "glue": {
        "raw":       "s3://f1-raw-satva/raw/kaggle",
        "processed": "s3://f1-processed-satva/processed/kaggle",
    },
}

RAW_BASE       = PATHS[MODE]["raw"]
PROCESSED_BASE = PATHS[MODE]["processed"]

# NOTE: No SparkSession block — Databricks provides `spark` automatically



def replace_backslash_n(df):
    """Replace the literal string '\\N' with null in every column."""
    for col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.when(F.col(col_name) == "\\N", F.lit(None)).otherwise(F.col(col_name)),
        )
    return df


@F.udf(LongType())
def parse_lap_time_ms(time_str):
    """
    Convert a lap-time string to milliseconds.
    Handles formats:
      "1:27.452"  → 87452
      "27.452"    → 27452  (no minutes)
    Returns None for null / unparseable input.
    """
    if time_str is None:
        return None
    try:
        if ":" in time_str:
            parts = time_str.split(":")
            minutes = int(parts[0])
            sec_parts = parts[1].split(".")
            seconds = int(sec_parts[0])
            millis = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return minutes * 60_000 + seconds * 1_000 + millis
        else:
            sec_parts = time_str.split(".")
            seconds = int(sec_parts[0])
            millis = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return seconds * 1_000 + millis
    except Exception:
        return None


def read_csv(table_name):
    """Read a raw CSV with header, all columns as string."""
    path = f"{RAW_BASE}/{table_name}.csv"
    print(f"  Reading {path}")
    return spark.read.csv(path, header=True, inferSchema=False)


def write_parquet(df, table_name, partition_cols=None):
    """Write a DataFrame as Parquet (optionally partitioned)."""
    path = f"{PROCESSED_BASE}/{table_name}"
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(path)
    count = df.count()
    print(f"  ✓ Wrote {table_name} → {path}  ({count:,} rows)")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TABLE CLEANERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def clean_circuits():
    print("[circuits]")
    df = read_csv("circuits")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("circuitId",  F.col("circuitId").cast(IntegerType()))
        .withColumn("lat",        F.col("lat").cast(DoubleType()))
        .withColumn("lng",        F.col("lng").cast(DoubleType()))
        .withColumn("alt",        F.col("alt").cast(IntegerType()))
        .dropDuplicates()
    )
    write_parquet(df, "circuits")


def clean_constructors():
    print("[constructors]")
    df = read_csv("constructors")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("constructorId", F.col("constructorId").cast(IntegerType()))
        .dropDuplicates()
    )
    write_parquet(df, "constructors")


def clean_drivers():
    print("[drivers]")
    df = read_csv("drivers")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("driverId",  F.col("driverId").cast(IntegerType()))
        .withColumn("number",    F.col("number").cast(IntegerType()))
        .withColumn("dob",       F.to_date(F.col("dob"), "yyyy-MM-dd"))
        .dropDuplicates()
    )
    write_parquet(df, "drivers")


def clean_seasons():
    print("[seasons]")
    df = read_csv("seasons")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("year", F.col("year").cast(IntegerType()))
        .dropDuplicates()
    )
    write_parquet(df, "seasons")


def clean_status():
    print("[status]")
    df = read_csv("status")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("statusId", F.col("statusId").cast(IntegerType()))
        .dropDuplicates()
    )
    write_parquet(df, "status")


def clean_races():
    print("[races]")
    df = read_csv("races")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("raceId",      F.col("raceId").cast(IntegerType()))
        .withColumn("year",        F.col("year").cast(IntegerType()))
        .withColumn("round",       F.col("round").cast(IntegerType()))
        .withColumn("circuitId",   F.col("circuitId").cast(IntegerType()))
        .withColumn("date",        F.to_date(F.col("date"), "yyyy-MM-dd"))
        .withColumn("fp1_date",    F.to_date(F.col("fp1_date"), "yyyy-MM-dd"))
        .withColumn("fp2_date",    F.to_date(F.col("fp2_date"), "yyyy-MM-dd"))
        .withColumn("fp3_date",    F.to_date(F.col("fp3_date"), "yyyy-MM-dd"))
        .withColumn("quali_date",  F.to_date(F.col("quali_date"), "yyyy-MM-dd"))
        .withColumn("sprint_date", F.to_date(F.col("sprint_date"), "yyyy-MM-dd"))
        .dropDuplicates()
    )
    write_parquet(df, "races", partition_cols=["year"])


def _races_lookup():
    """Helper: load raceId → year mapping for joins."""
    return (
        read_csv("races")
        .transform(replace_backslash_n)
        .select(
            F.col("raceId").cast(IntegerType()).alias("raceId"),
            F.col("year").cast(IntegerType()).alias("year"),
        )
    )


def clean_results():
    print("[results]")
    df = read_csv("results")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("resultId",          F.col("resultId").cast(IntegerType()))
        .withColumn("raceId",            F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",          F.col("driverId").cast(IntegerType()))
        .withColumn("constructorId",     F.col("constructorId").cast(IntegerType()))
        .withColumn("number",            F.col("number").cast(IntegerType()))
        .withColumn("grid",              F.col("grid").cast(IntegerType()))
        .withColumn("position",          F.col("position").cast(IntegerType()))
        .withColumn("positionOrder",     F.col("positionOrder").cast(IntegerType()))
        .withColumn("points",            F.col("points").cast(DoubleType()))
        .withColumn("laps",              F.col("laps").cast(IntegerType()))
        .withColumn("milliseconds",      F.col("milliseconds").cast(LongType()))
        .withColumn("fastestLap",        F.col("fastestLap").cast(IntegerType()))
        .withColumn("rank",              F.col("rank").cast(IntegerType()))          
        .withColumn("fastestLapTime_ms", parse_lap_time_ms(F.col("fastestLapTime")))
        .withColumn("fastestLapSpeed",   F.col("fastestLapSpeed").cast(DoubleType()))  
        .withColumn("statusId",          F.col("statusId").cast(IntegerType()))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "results", partition_cols=["year"])


def clean_qualifying():
    print("[qualifying]")
    df = read_csv("qualifying")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("qualifyId",     F.col("qualifyId").cast(IntegerType()))
        .withColumn("raceId",        F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",      F.col("driverId").cast(IntegerType()))
        .withColumn("constructorId", F.col("constructorId").cast(IntegerType()))
        .withColumn("number",        F.col("number").cast(IntegerType()))
        .withColumn("position",      F.col("position").cast(IntegerType()))
        .withColumn("q1_ms",         parse_lap_time_ms(F.col("q1")))
        .withColumn("q2_ms",         parse_lap_time_ms(F.col("q2")))
        .withColumn("q3_ms",         parse_lap_time_ms(F.col("q3")))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "qualifying", partition_cols=["year"])


def clean_lap_times():
    print("[lap_times]")
    df = read_csv("lap_times")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("raceId",       F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",     F.col("driverId").cast(IntegerType()))
        .withColumn("lap",          F.col("lap").cast(IntegerType()))
        .withColumn("position",     F.col("position").cast(IntegerType()))
        .withColumn("milliseconds", F.col("milliseconds").cast(LongType()))
        .withColumn("time_ms",      parse_lap_time_ms(F.col("time")))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "lap_times", partition_cols=["year"])


def clean_pit_stops():
    print("[pit_stops]")
    df = read_csv("pit_stops")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("raceId",       F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",     F.col("driverId").cast(IntegerType()))
        .withColumn("stop",         F.col("stop").cast(IntegerType()))
        .withColumn("lap",          F.col("lap").cast(IntegerType()))
        .withColumn("milliseconds", F.col("milliseconds").cast(LongType()))
        .withColumn("duration_ms",   parse_lap_time_ms(F.col("duration")))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "pit_stops", partition_cols=["year"])


def clean_driver_standings():
    print("[driver_standings]")
    df = read_csv("driver_standings")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("driverStandingsId", F.col("driverStandingsId").cast(IntegerType()))
        .withColumn("raceId",            F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",          F.col("driverId").cast(IntegerType()))
        .withColumn("points",            F.col("points").cast(DoubleType()))
        .withColumn("position",          F.col("position").cast(IntegerType()))
        .withColumn("wins",              F.col("wins").cast(IntegerType()))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "driver_standings", partition_cols=["year"])


def clean_constructor_standings():
    print("[constructor_standings]")
    df = read_csv("constructor_standings")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("constructorStandingsId", F.col("constructorStandingsId").cast(IntegerType()))
        .withColumn("raceId",                 F.col("raceId").cast(IntegerType()))
        .withColumn("constructorId",          F.col("constructorId").cast(IntegerType()))
        .withColumn("points",                 F.col("points").cast(DoubleType()))
        .withColumn("position",               F.col("position").cast(IntegerType()))
        .withColumn("wins",                   F.col("wins").cast(IntegerType()))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "constructor_standings", partition_cols=["year"])


def clean_constructor_results():
    print("[constructor_results]")
    df = read_csv("constructor_results")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("constructorResultsId", F.col("constructorResultsId").cast(IntegerType()))
        .withColumn("raceId",               F.col("raceId").cast(IntegerType()))
        .withColumn("constructorId",        F.col("constructorId").cast(IntegerType()))
        .withColumn("points",               F.col("points").cast(DoubleType()))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "constructor_results", partition_cols=["year"])


def clean_sprint_results():
    print("[sprint_results]")
    df = read_csv("sprint_results")
    df = replace_backslash_n(df)
    df = (
        df
        .withColumn("resultId",          F.col("resultId").cast(IntegerType()))
        .withColumn("raceId",            F.col("raceId").cast(IntegerType()))
        .withColumn("driverId",          F.col("driverId").cast(IntegerType()))
        .withColumn("constructorId",     F.col("constructorId").cast(IntegerType()))
        .withColumn("number",            F.col("number").cast(IntegerType()))
        .withColumn("grid",              F.col("grid").cast(IntegerType()))
        .withColumn("position",          F.col("position").cast(IntegerType()))
        .withColumn("positionOrder",     F.col("positionOrder").cast(IntegerType()))
        .withColumn("points",            F.col("points").cast(DoubleType()))
        .withColumn("laps",              F.col("laps").cast(IntegerType()))
        .withColumn("milliseconds",      F.col("milliseconds").cast(LongType()))
        .withColumn("fastestLap",        F.col("fastestLap").cast(IntegerType()))
        .withColumn("fastestLapTime_ms", parse_lap_time_ms(F.col("fastestLapTime")))
        .withColumn("statusId",          F.col("statusId").cast(IntegerType()))
        .dropDuplicates()
    )
    df = df.join(_races_lookup(), on="raceId", how="left")
    write_parquet(df, "sprint_results", partition_cols=["year"])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  MAIN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print("=" * 60)
    print("clean_kaggle.py — Starting")
    print(f"  MODE:      {MODE}")
    print(f"  RAW:       {RAW_BASE}")
    print(f"  PROCESSED: {PROCESSED_BASE}")
    print("=" * 60)

    # Reference tables (no partitioning)
    clean_circuits()
    clean_constructors()
    clean_drivers()
    clean_seasons()
    clean_status()

    # Fact / event tables (partitioned by year)
    clean_races()
    clean_results()
    clean_qualifying()
    clean_lap_times()
    clean_pit_stops()
    clean_driver_standings()
    clean_constructor_standings()
    clean_constructor_results()
    clean_sprint_results()

    print("=" * 60)
    print("clean_kaggle.py — Complete ✓")
    print("=" * 60)


# Call main directly — no if __name__ block needed in Databricks
main()

## Cell 4 — Verify output
Run after main() completes to confirm all 14 tables were written.

In [0]:
tables = dbutils.fs.ls("/Volumes/workspace/f1_pipeline/processed_kaggle/")
print(f"Tables written: {len(tables)} (expecting 14)\n")
for t in tables:
    print(f"  📁 {t.name}")

## Cell 5 — Spot check: drivers table

In [0]:
df = spark.read.parquet("/Volumes/workspace/f1_pipeline/processed_kaggle/drivers")
print(f"Total rows: {df.count():,}")
df.printSchema()
display(df.limit(10))

driverId,driverRef,number,code,forename,surname,dob,nationality,url
10,glock,null,GLO,Timo,Glock,1982-03-18,German,http://en.wikipedia.org/wiki/Timo_Glock
54,burti,null,null,Luciano,Burti,1975-03-05,Brazilian,http://en.wikipedia.org/wiki/Luciano_Burti
83,lamy,null,null,Pedro,Lamy,1972-03-20,Portuguese,http://en.wikipedia.org/wiki/Pedro_Lamy
87,blundell,null,null,Mark,Blundell,1966-04-08,British,http://en.wikipedia.org/wiki/Mark_Blundell
118,warwick,null,null,Derek,Warwick,1954-08-27,British,http://en.wikipedia.org/wiki/Derek_Warwick
181,manfred_winkelhock,null,null,Manfred,Winkelhock,1951-10-06,German,http://en.wikipedia.org/wiki/Manfred_Winkelhock
182,lauda,null,null,Niki,Lauda,1949-02-22,Austrian,http://en.wikipedia.org/wiki/Niki_Lauda
254,zorzi,null,null,Renzo,Zorzi,1946-12-12,Italian,http://en.wikipedia.org/wiki/Renzo_Zorzi
260,andersson,null,null,Conny,Andersson,1939-12-28,Swedish,http://en.wikipedia.org/wiki/Conny_Andersson_(racing_driver)
261,dryver,null,null,Bernard,de Dryver,1952-09-19,Belgian,http://en.wikipedia.org/wiki/Bernard_de_Dryver


## Cell 6 — Spot check: Hamilton results

In [0]:
results = spark.read.parquet("/Volumes/workspace/f1_pipeline/processed_kaggle/results")
hamilton = results.filter(results.driverId == 1)
print(f"Hamilton results: {hamilton.count()} rows")
display(hamilton.select("year", "position", "points", "fastestLapTime_ms").orderBy("year").limit(20))

year,position,points,fastestLapTime_ms
2007,1,10.0,73222
2007,2,8.0,94270
2007,2,8.0,82876
2007,1,10.0,80171
2007,3,6.0,81675
2007,2,8.0,96701
2007,7,2.0,72506
2007,1,10.0,88193
2007,4,5.0,108215
2007,3,6.0,86351


## Cell 7 — Null audit: confirm no \\N strings remain

In [0]:
df = spark.read.parquet("/Volumes/workspace/f1_pipeline/processed_kaggle/drivers")
print("Checking drivers table for leftover \\N strings...")
found_any = False
for c in df.columns:
    count = df.filter(F.col(c).cast("string") == "\\N").count()
    if count > 0:
        print(f"  ⚠ {c}: {count} \\N values remaining!")
        found_any = True
if not found_any:
    print("  ✓ All columns clean — no \\N strings found")


# Verification Checks : Not to be copied in glue 

In [0]:
tables = [
    "circuits", "constructors", "drivers", "seasons", "status",
    "races", "results", "qualifying", "lap_times", "pit_stops",
    "driver_standings", "constructor_standings", "constructor_results", "sprint_results"
]

print(f"{'Table':<30} {'Rows':>10}  {'Columns':>8}")
print("─" * 52)
total = 0
for t in tables:
    df = spark.read.parquet(f"/Volumes/workspace/f1_pipeline/processed_kaggle/{t}")
    count = df.count()
    cols = len(df.columns)
    total += count
    print(f"  {t:<28} {count:>10,}  {cols:>8}")
print("─" * 52)
print(f"  {'TOTAL':<28} {total:>10,}")


In [0]:
# Should show integers, doubles, dates — NOT all strings
for t in ["drivers", "results", "races"]:
    print(f"\n{'='*40}\n  {t}\n{'='*40}")
    spark.read.parquet(f"/Volumes/workspace/f1_pipeline/processed_kaggle/{t}").printSchema()


In [0]:
# Verify parse_lap_time_ms on known results
results = spark.read.parquet("/Volumes/workspace/f1_pipeline/processed_kaggle/results")

# Pick a known result: 2008 Australian GP, Hamilton, fastestLapTime "1:27.452" should = 87452 ms
check = results.filter(
    (results.resultId == 1)
).select("fastestLapTime", "fastestLapTime_ms")

display(check)
# Expected: fastestLapTime = "1:27.452", fastestLapTime_ms = 87452


fastestLapTime,fastestLapTime_ms
1:27.452,87452


In [0]:
tables = [
    "circuits", "constructors", "drivers", "seasons", "status",
    "races", "results", "qualifying", "lap_times", "pit_stops",
    "driver_standings", "constructor_standings", "constructor_results", "sprint_results"
]

print("Checking all tables for leftover \\N strings...\n")
all_clean = True
for t in tables:
    df = spark.read.parquet(f"/Volumes/workspace/f1_pipeline/processed_kaggle/{t}")
    for c in df.columns:
        count = df.filter(F.col(c).cast("string") == "\\N").count()
        if count > 0:
            print(f"  ⚠ {t}.{c}: {count} \\N values")
            all_clean = False

if all_clean:
    print("  ✓ All 14 tables clean — zero \\N strings found")
